In [ ]:
"""
topic: /joint_states 
msg: sensor_msgs/JointState
msg example:
    header:
        stamp:
            sec: 1783090310
            nanosec: 659695359
        frame_id: ''
    name:
    - base_to_cylinder_joint
    - cylinder_to_arm1_joint
    - arm_link1_to_arm2_joint
    - arm_link2_to_arm3_joint
    - arm_link3_to_arm4_joint
    - arm_link4_to_armFork_joint
    - basket_link_to_armFork_joint
    - basket_crane_link_to_basket_link
    - winch_link_to_basket_crane_link
    position:
    - -0.026179936699320342
    - 0.03490658503988659
    - 3.0268351236983904e-05
    - 3.0268351236983904e-05
    - 3.0268351236983904e-05
    - 0.050093414960113415
    - -0.034988610401461034
    - 0.0
    - 0.08344268798828125
    velocity: []
    effort: []

"""

"""
topic: /move_group/display_controller_planned_path
msg: moveit_msgs/DisplayTrajectory
msg example:
model_id: ''                                                                                                                                                                                                                 14:59:14 [278/279]
trajectory:                                                         
- joint_trajectory:                                                 
    header:                                                         
      stamp:                                                        
        sec: 0                                                      
        nanosec: 0                                                  
      frame_id: base_link                                           
    joint_names:                                                    
    - base_to_cylinder_joint                                        
    - cylinder_to_arm1_joint                                        
    - arm_link1_to_arm2_joint                                       
    - arm_link2_to_arm3_joint                                       
    - arm_link3_to_arm4_joint                                       
    - arm_link4_to_armFork_joint                                    
    points:                                                         
    - positions:                                                    
      - -0.026179936699320342                                       
      - -0.008726646259971648                                       
      - 3.0268351236983904e-05                                      
      - 3.0268351236983904e-05                                      
      - 3.0268351236983904e-05                                      
      - 0.09372664625997165                                         
      velocities: []                                                
      accelerations: []                                             
      effort: []                                                    
      time_from_start:                                              
        sec: 0                                                      
        nanosec: 0                                                  
    - positions:                                                    
      - -0.06735381502054213                                        
      - -0.031275628424015524                                       
      - -0.03900616398622193                                        
      - -0.038718939124295665                                       
      - -0.039276215595813645                                       
      - 0.0795991791287562                                          
      velocities: []                                                
      accelerations: []                                             
      effort: []                                                    
      time_from_start:                                              
        sec: 0                                                      
        nanosec: 0    
  multi_dof_joint_trajectory:
    header:
      stamp:
        sec: 0
        nanosec: 0
      frame_id: ''
    joint_names: []
    points: []
trajectory_start:
  joint_state:
    header:
      stamp:
        sec: 0
        nanosec: 0
      frame_id: ''
    name: []
    position: []
    velocity: []
    effort: []
  multi_dof_joint_state:
    header:
      stamp:
        sec: 0
        nanosec: 0
      frame_id: ''
    joint_names: []
    transforms: []
    twist: []
    wrench: []
  attached_collision_objects: []
  is_diff: false

"""

In [ ]:
import rclpy
import rclpy.logging
import rclpy.time
from rclpy.node import Node

if not rclpy.ok():
    rclpy.init()

    node = Node('joint_tracking')
    node.get_logger().info("Starting joint tracking node")

In [ ]:
import math
import os

from builtin_interfaces.msg import Time
from sensor_msgs.msg import JointState
from std_msgs.msg import Float32MultiArray
from geometry_msgs.msg import PoseStamped
from telehandler_moveit.action import MoveToPosition
from action_msgs.msg import GoalStatus
from rclpy.action import ActionClient

# Match the variable names exactly to what your callbacks update
joint_state: JointState | None = None
joints_values: Float32MultiArray | None = None
encoder_values: Float32MultiArray | None = None


def get_ros_timestamp(node) -> Time:
    return node.get_clock().now().to_msg()

In [ ]:
import os
import csv

output_csv = str(os.environ.get('HOME'))+"/mnt/telehandler/criarte_ws/scripts/Fortis_KPI/TE02_001/kpi_results.csv"


def create_csv():
    if not os.path.exists(os.path.dirname(output_csv)):
        os.makedirs(os.path.dirname(output_csv))

    with open(output_csv, 'w', newline='') as csvfile:
        fieldnames = ['timestamp', 'base_to_cylinder_error', 'cylinder_to_arm1_error', 'arm_link1_to_arm2_error',
                      'arm_link2_to_arm3_error', 'arm_link3_to_arm4_error', 'arm_link4_to_armFork_error', 
                      "basket_link_to_armFork_error", "basket_crane_link_to_basket_error", "winch_link_to_basket_crane_error"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()


def write_csv(timestamp, can_positions, ros_positions):
    errors = [
        ros_positions[i] - can_positions[i]
        for i in range(9)
    ]

    with open(output_csv, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        row_data = [f"{timestamp.sec}.{timestamp.nanosec:09d}"] + errors
        writer.writerow(row_data)


create_csv()

In [ ]:
def client_feedback(feedback_msg):
    global joint_state, encoder_values, joints_values

    if joint_state is None or joints_values is None or encoder_values is None:
        node.get_logger().warn("Awaiting JointState, Encoders, and CAN Joint data...",
                                throttle_duration_sec=1.0)
        return

    timestamp = get_ros_timestamp(node)

    # Safely extract the raw lists from the ROS messages
    j_data = joints_values.data
    e_data = encoder_values.data

    # --- HARDWARE TO ROS MAPPING ---
    # The telehandler boom extends 3 physical joints using 1 measured cylinder
    boom_segment_extension = j_data[1] / 3.0

    # Construct an array of exactly 6 values to match the ROS joint_state.
    # NOTE: I have assigned E[0], E[1] and J[2] as placeholders for the other
    # joints. You MUST verify that these indices map to the correct mechanical joints!
    can_positions = [
        j_data[0],               # Index 0: base_to_cylinder_joint
        j_data[1],               # Index 1: cylinder_to_arm1_joint
        boom_segment_extension,  # Index 2: arm_link1_to_arm2_joint
        boom_segment_extension,  # Index 3: arm_link2_to_arm3_joint
        boom_segment_extension,  # Index 4: arm_link3_to_arm4_joint
        j_data[2],               # Index 5: arm_link4_to_armFork_joint
        e_data[0],               # Index 6: basket_link_to_armFork_joint
        0.0,                     # Index 7: basket_crane_link_to_basket_joint
        e_data[1],               # Index 8: winch_link_to_basket_cr
    ]

    # Ensure we only grab the first 6 elements of the ROS target path,
    # ignoring wheels, basket, or other joints that might be in the URDF.
    ros_positions = joint_state.position[:9]

    # Pass the perfectly aligned arrays to the CSV writer
    write_csv(timestamp, can_positions, ros_positions)


def send_action_goal(node):
    _action_client = ActionClient(
        node, MoveToPosition, '/action/MoveToPosition')
    try:
        node.get_logger().info("Waiting for action server...")
        if not _action_client.wait_for_server(timeout_sec=10.0):
            node.get_logger().error("Action server timeout.")
            return 1

        goal_msg = MoveToPosition.Goal()
        goal_msg.target_pose.header.frame_id = "base_link"
        goal_msg.target_pose.pose.position.x = 3.5
        goal_msg.target_pose.pose.position.y = -4.3
        goal_msg.target_pose.pose.position.z = 5.0
        goal_msg.target_pose.pose.orientation.w = 1.0

        node.get_logger().info("Sending goal...")
        send_goal_future = _action_client.send_goal_async(
            goal_msg, feedback_callback=client_feedback)

        rclpy.spin_until_future_complete(node, send_goal_future)

        goal_handle = send_goal_future.result()
        if not goal_handle.accepted:
            node.get_logger().error("Goal rejected by server.")
            return 1

        node.get_logger().info("Goal accepted! Waiting for execution...")
        result_future = goal_handle.get_result_async()
        rclpy.spin_until_future_complete(node, result_future)

        wrapper_result = result_future.result()

        if wrapper_result.status == GoalStatus.STATUS_SUCCEEDED:
            node.get_logger().info("Goal succeeded!")
            return 0
        else:
            node.get_logger().warn(
                f"Goal failed. Status: {wrapper_result.status}")
            return 1
    finally:
        _action_client.destroy()

In [ ]:
def state_cb(msg):
    global joint_state
    joint_state = msg


def encoder_cb(msg):
    global encoder_values
    encoder_values = msg


def joints_cb(msg):
    global joints_values
    joints_values = msg

In [ ]:
def main():
    node.create_subscription(JointState, '/joint_states', state_cb, 10)
    node.create_subscription(Float32MultiArray, '/encoders', encoder_cb, 10)
    node.create_subscription(Float32MultiArray, '/joints', joints_cb, 10)
    
    send_action_goal(node)
    rclpy.spin(node)

In [ ]:
"""TEST FUNCTIONS"""
main()